# Lecture: Building a Reproducible Analysis Workflow

In this lecture, we will organize an analysis around a question. As each step creates a need, we will introduce the Python and Pandas tools required to continue.


## Learning goals

By the end of this lecture, you should be able to:

- organize an analysis using a reproducible workflow;
- identify the observations and features needed to answer a question;
- convert text to Pandas date-time values;
- use date-time methods to create a new feature;
- explain scalar broadcasting and vectorized operations;
- group observations and calculate a summary for each group;
- filter observations using Boolean conditions and `.loc`;
- check whether an analytical result is reasonable; and
- distinguish evidence, conclusions, and limitations.


## The reproducible analysis workflow

We will use seven steps:

1. **Question:** What do we want to learn?
2. **Data:** Which observations and features can help answer it?
3. **Operation:** What should the code select, calculate, or compare?
4. **Check:** Does the result have the expected observations, values, and units?
5. **Evidence:** Which result directly answers the question?
6. **Conclusion:** What claim is supported?
7. **Limitation:** What should we avoid concluding?


Code that runs without an error can still answer the wrong question. A reproducible analysis preserves the code and records how each result supports the conclusion.


## Step 1: Question

This lecture uses a snapshot of worldwide seismic events recorded by the U.S. Geological Survey from January through June 2026. The file includes events with a reported magnitude of at least 2.5.

### Which month had the highest percentage of recorded magnitude-5.0+ events?


### Discussion: choose a fair comparison

Why might a percentage provide a fairer monthly comparison than a count alone?


## Step 2: Data

Each observation represents one recorded seismic event. We will need:

- `event_time` to identify the month;
- `magnitude` to identify magnitude-5.0+ events; and
- `event_id` to count the recorded events.


### Data dictionary

Before writing the analysis, review all the features available in the file.

| Feature | Feature type | Description |
| --- | --- | --- |
| `event_id` | Identifier | Distinguishes one event record from another |
| `event_time` | Date-time | Date and time when the event began, recorded in UTC |
| `magnitude` | Quantitative | Estimate of the event's size |
| `depth_km` | Quantitative | Estimated depth where rupture began, in kilometers |
| `magnitude_type` | Categorical | Method used to calculate magnitude |
| `reporting_network` | Categorical | Network that supplied the preferred event information |
| `event_type` | Categorical | Classification such as earthquake or mining explosion |
| `review_status` | Categorical | Whether the record was reviewed or remained automatic |
| `place` | Text | Description of the event's location |
| `horizontal_error_km` | Quantitative | Estimate of horizontal location uncertainty, in kilometers |


### Load and inspect the data


In [1]:
import pandas as pd

earthquakes = pd.read_csv('data/usgs_earthquakes_2026_h1_m25.csv')



In [2]:
earthquakes.head()

,event_id,event_time,magnitude,depth_km,magnitude_type,reporting_network,event_type,review_status,place,horizontal_error_km
0,us7000rlt7,2026-01-01T00:03:02.325Z,3.2,5.000,ml,us,earthquake,reviewed,"100 km N of Yakutat, Alaska",3.28
1,us7000rlt9,2026-01-01T00:46:26.988Z,3.9,10.000,mb,us,earthquake,reviewed,"262 km SE of Chiniak, Alaska",6.92
2,ak2026aaccmm,2026-01-01T01:03:04.655Z,2.9,3.700,ml,ak,earthquake,reviewed,"93 km SE of King Cove, Alaska",5.80
3,us7000rrd6,2026-01-01T01:25:20.926Z,4.5,118.207,mb,us,earthquake,reviewed,South Sandwich Islands region,14.01
4,us7000rltg,2026-01-01T01:53:01.809Z,6.0,10.000,mww,us,earthquake,reviewed,southeast Indian Ridge,10.14


In [3]:
earthquakes.tail()

,event_id,event_time,magnitude,depth_km,magnitude_type,reporting_network,event_type,review_status,place,horizontal_error_km
14133,us6000t9ak,2026-06-30T21:49:19.744Z,4.7,10.000,mb,us,earthquake,reviewed,Federated States of Micronesia region,10.81
14134,us6000tatb,2026-06-30T22:45:15.182Z,2.6,65.730,ml,us,earthquake,reviewed,"40 km S of Unalaska, Alaska",6.89
14135,aka2026mwglzp,2026-06-30T23:15:11.489Z,2.8,50.600,ml,ak,earthquake,reviewed,"18 km N of Sterling, Alaska",1.90
14136,us6000t9bc,2026-06-30T23:40:16.981Z,4.8,45.088,mww,us,earthquake,reviewed,"10 km W of Hyūga, Japan",5.85
14137,us6000t9bg,2026-06-30T23:44:54.898Z,5.3,10.000,mww,us,earthquake,reviewed,"272 km SSE of Dunhuang, China",9.38


In [4]:
earthquakes.info()

<class 'pandas.DataFrame'>
RangeIndex: 14138 entries, 0 to 14137
Data columns (total 10 columns):
 #   Column               Non-Null Count  Dtype  
---  ------               --------------  -----  
 0   event_id             14138 non-null  str    
 1   event_time           14138 non-null  str    
 2   magnitude            14138 non-null  float64
 3   depth_km             14138 non-null  float64
 4   magnitude_type       14138 non-null  str    
 5   reporting_network    14138 non-null  str    
 6   event_type           14138 non-null  str    
 7   review_status        14138 non-null  str    
 8   place                14138 non-null  str    
 9   horizontal_error_km  13854 non-null  float64
dtypes: float64(3), str(7)
memory usage: 1.1 MB


In [5]:
earthquakes.isnull().sum()

event_id                 0
event_time               0
magnitude                0
depth_km                 0
magnitude_type           0
reporting_network        0
event_type               0
review_status            0
place                    0
horizontal_error_km    284
dtype: int64

### Discussion: check the data

1. What does each row represent?
2. Does the file contain the features needed to answer the question?
3. How is `event_time` currently stored?
4. Are either `event_time`, `magnitude`, or `event_id` missing any values?


## Step 3: Operation

To answer the question, we need to:

1. convert `event_time` from text to date-time values;
2. identify the month of each event;
3. identify which events had magnitudes of at least 5.0;
4. group the observations by month;
5. count all events and magnitude-5.0+ events in each month; and
6. calculate a monthly percentage.


### Convert the event times

`pd.to_datetime()` converts the text values to date-time values. `utc=True` retains the UTC time-zone information indicated by the timestamps.


In [8]:
event_time = pd.to_datetime(earthquakes['event_time'], utc=True)
event_time.head()

0   2026-01-01 00:03:02.325000+00:00
1   2026-01-01 00:46:26.988000+00:00
2   2026-01-01 01:03:04.655000+00:00
3   2026-01-01 01:25:20.926000+00:00
4   2026-01-01 01:53:01.809000+00:00
Name: event_time, dtype: datetime64[us, UTC]

The `.dt` accessor provides date-time operations for a Series. Here, `.dt.month_name()` extracts the name of the month from every timestamp.


In [11]:
event_months = event_time.dt.month_name()
event_months.head()

0    January
1    January
2    January
3    January
4    January
Name: event_time, dtype: str

In [12]:
event_months.value_counts()

event_time
April       2618
March       2455
June        2426
January     2290
May         2219
February    2130
Name: count, dtype: int64

### Identify magnitude-5.0+ events

The comparison below asks whether each value in the `magnitude` Series is at least 5.0.


In [13]:
magnitude_5_plus = earthquakes['magnitude'] >=5.0
magnitude_5_plus.head()

0    False
1    False
2    False
3    False
4     True
Name: magnitude, dtype: bool

Pandas **broadcasts** the single value `5.0` across the entire Series and performs the comparison for every observation. This is a **vectorized operation**: we operate on the Series without writing a loop for its individual values.


### Add the features needed for grouping

Create a separate DataFrame so that the original imported data remains unchanged.


In [17]:
analysis_data = earthquakes.copy()

analysis_data.to_csv('analysis_data.csv', index=False)
analysis_data['event_month'] = event_months
analysis_data['magnitude_5_plus'] = magnitude_5_plus

analysis_data.head()

,event_id,event_time,magnitude,depth_km,magnitude_type,reporting_network,event_type,review_status,place,horizontal_error_km,event_month,magnitude_5_plus
0,us7000rlt7,2026-01-01T00:03:02.325Z,3.2,5.000,ml,us,earthquake,reviewed,"100 km N of Yakutat, Alaska",3.28,January,False
1,us7000rlt9,2026-01-01T00:46:26.988Z,3.9,10.000,mb,us,earthquake,reviewed,"262 km SE of Chiniak, Alaska",6.92,January,False
2,ak2026aaccmm,2026-01-01T01:03:04.655Z,2.9,3.700,ml,ak,earthquake,reviewed,"93 km SE of King Cove, Alaska",5.80,January,False
3,us7000rrd6,2026-01-01T01:25:20.926Z,4.5,118.207,mb,us,earthquake,reviewed,South Sandwich Islands region,14.01,January,False
4,us7000rltg,2026-01-01T01:53:01.809Z,6.0,10.000,mww,us,earthquake,reviewed,southeast Indian Ridge,10.14,January,True


### Group and summarize the observations

`.groupby("event_month")` forms one group for each month. `.agg()` then calculates two summaries for every group:

- `count` counts all event identifiers; and
- `sum` counts the `True` values in `magnitude_5_plus` because `True` is evaluated as 1.


In [19]:
analysis_data[ ['magnitude', 'magnitude_5_plus'] ].head(25)

,magnitude,magnitude_5_plus
0,3.2,False
1,3.9,False
2,2.9,False
3,4.5,False
4,6.0,True
5,2.9,False
6,4.6,False
7,4.4,False
8,2.9,False
9,4.5,False


In [32]:
monthly_summary = (analysis_data
        .groupby('event_month')
        .agg(event_count=('event_id', 'count'),
             magnitude_5_plus_count=('magnitude_5_plus', 'sum')
            )
    )

monthly_summary

,event_count,magnitude_5_plus_count
event_month,,
April,2618,169
February,2130,113
January,2290,179
June,2426,180
March,2455,159
May,2219,118


### Calculate the percentage

Divide each month's magnitude-5.0+ count by its total event count and multiply by 100. These arithmetic operations are vectorized across the two columns.


In [33]:
monthly_summary['magnitude_5_plus_percent'] = (monthly_summary['magnitude_5_plus_count']
                                               / monthly_summary['event_count']) * 100

monthly_summary.head()

,event_count,magnitude_5_plus_count,magnitude_5_plus_percent
event_month,,,
April,2618,169,6.455309
February,2130,113,5.305164
January,2290,179,7.816594
June,2426,180,7.419621
March,2455,159,6.476578


Sort the months from the greatest percentage to the smallest so the result that answers the question appears first.


In [38]:
monthly_summary.sort_values(by='magnitude_5_plus_percent', ascending=False, inplace=True)
monthly_summary

,event_count,magnitude_5_plus_count,magnitude_5_plus_percent
event_month,,,
January,2290,179,7.816594
June,2426,180,7.419621
March,2455,159,6.476578
April,2618,169,6.455309
May,2219,118,5.317711
February,2130,113,5.305164


## Step 4: Check

A sensible result should satisfy all of the following conditions:

- the table contains six months;
- the monthly event counts add to the number of observations in the original DataFrame;
- the monthly magnitude-5.0+ counts add to the total number of `True` values; and
- every percentage falls between 0 and 100.


### Discussion: check the result

Did the result pass all four checks?


## Step 5: Evidence

Display the counts and percentage needed to answer the question.


In [39]:
monthly_summary

,event_count,magnitude_5_plus_count,magnitude_5_plus_percent
event_month,,,
January,2290,179,7.816594
June,2426,180,7.419621
March,2455,159,6.476578
April,2618,169,6.455309
May,2219,118,5.317711
February,2130,113,5.305164


### Discussion: identify the evidence

Which row provides the evidence needed to answer the question?


magnitude_5_plus_percent column

## Step 6: Conclusion

### Discussion: form a conclusion

Which month had the highest percentage of recorded magnitude-5.0+ events, and what was that percentage?


January

## Step 7: Limitation

### Discussion: identify a limitation

Does this result establish that earthquakes are generally stronger in January?


No it just shows that earthquakes happen more frequent in Janurary

## A second workflow example

We can apply similar techniques to a different question without inspecting the entire DataFrame again.


## Step 1: Question

### On average in our dataset, which were deeper: earthquakes or mining explosions?


## Step 2: Data

We need `event_type` to identify earthquakes and mining explosions, `depth_km` to calculate their average depths, and `event_id` to count the observations in each group.

Select the two event types needed for the comparison. `.isin()` creates a Boolean Series that is `True` when a value matches one of the values in the list.


### Discussion: check the selected groups

Which event types are present, and how do their group sizes compare?


## Step 3: Operation

Group the selected observations by `event_type`. For each group, calculate the number of observations and the mean depth.


The result has one row for each event type. `event_count` records the group size, and `mean_depth_km` records its average depth in kilometers.


## Step 4: Check

Check that the group counts add to the number of selected observations and that neither group has missing depth values.


### Discussion: check the result

Did the result pass the checks?


## Step 5: Evidence

The `depth_summary` table displays the number of observations and mean depth for both event types.


## Step 6: Conclusion

### Discussion: form a conclusion

On average in this dataset, which event type was deeper, and what were the two mean depths?


## Step 7: Limitation

### Discussion: identify a limitation

What should we keep in mind when interpreting this comparison?


## Discussion: review

1. Why did the analysis compare percentages instead of only monthly counts?
2. What did scalar broadcasting accomplish when we compared `magnitude` with `5.0`?
3. Why did summing the Boolean feature count magnitude-5.0+ events?
4. Why did we check the grouped counts against the original data?
5. What did `.isin()` accomplish in the depth comparison?
6. Why did the summary include group counts as well as mean depths?
7. Why should the mining-explosion mean be interpreted cautiously?


## Summary

We used the analysis workflow to answer two related questions without repeating unnecessary inspection. Along the way, we converted date-time values, extracted months, used scalar broadcasting and vectorized operations, filtered observations with `.loc`, grouped observations, calculated counts, percentages, and means, checked results, identified evidence, formed conclusions, and stated limitations.
